In [2]:
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import MapType, StringType
from pyspark.sql.functions import col 
from pyspark.sql.types import StructField, StructType 

In [3]:
#Se importa la sesion de Spark
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName('LecturaArchivosCSV') \
    .getOrCreate()

path_archivo = 'databses/vgsales.csv'

#Generacion del dataframe con Spark
#Los encabezados se estan tomando como si fueran datos, hay que usar header
df =spark.read.csv(path_archivo, header = True)

In [4]:
df.show()

+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|   11.27|    8.89|   10.22|  

In [5]:
#Generacon de Comandos SQL usando spar.sql
#Con esto no se deben usar las funciones de SPark
#Alternativa válida
df.createOrReplaceTempView('VGSALES')
sql_str = 'select Publisher, sum(NA_Sales), sum(Global_Sales), sum(JP_Sales) from VGSALES group by Genre, Publisher order by Publisher'
spark.sql(sql_str).show()

+--------------------+------------------+-------------------+-------------+
|           Publisher|     sum(NA_Sales)|  sum(Global_Sales)|sum(JP_Sales)|
+--------------------+------------------+-------------------+-------------+
|     10TACLE Studios|               0.0|               0.03|          0.0|
|     10TACLE Studios|              0.01|               0.02|          0.0|
|     10TACLE Studios|              0.06|               0.06|          0.0|
|          1C Company|               0.0|               0.04|          0.0|
|          1C Company|              0.01|               0.05|          0.0|
|          1C Company|               0.0|               0.01|          0.0|
|20th Century Fox ...|              0.21|               0.22|          0.0|
|20th Century Fox ...|              1.61| 1.7200000000000002|          0.0|
|              2D Boy|               0.0|               0.04|          0.0|
|                 3DO|              0.06|               0.13|          0.0|
|           

In [6]:
#Redondear las columnas 
#Cambio de nombres
df.createOrReplaceTempView('VGSALES')
sql_str = 'select Publisher, round(sum(NA_Sales)) as TotalNA, round(sum(Global_Sales)) as TotalGl from VGSALES group by Publisher order by 3 desc'
spark.sql(sql_str).show(10,80) #Lineas y caracteres

+----------------------------+-------+-------+
|                   Publisher|TotalNA|TotalGl|
+----------------------------+-------+-------+
|                    Nintendo|  817.0| 1787.0|
|             Electronic Arts|  595.0| 1110.0|
|                  Activision|  430.0|  727.0|
| Sony Computer Entertainment|  265.0|  607.0|
|                     Ubisoft|  253.0|  475.0|
|        Take-Two Interactive|  220.0|  400.0|
|                         THQ|  209.0|  341.0|
|Konami Digital Entertainment|   92.0|  284.0|
|                        Sega|  109.0|  273.0|
|          Namco Bandai Games|   70.0|  254.0|
+----------------------------+-------+-------+
only showing top 10 rows


In [7]:
#Muestra las primeras 5 filas, con un maximo de 40 carateres por columnas
spark.sql(sql_str).show(5,40)

+---------------------------+-------+-------+
|                  Publisher|TotalNA|TotalGl|
+---------------------------+-------+-------+
|                   Nintendo|  817.0| 1787.0|
|            Electronic Arts|  595.0| 1110.0|
|                 Activision|  430.0|  727.0|
|Sony Computer Entertainment|  265.0|  607.0|
|                    Ubisoft|  253.0|  475.0|
+---------------------------+-------+-------+
only showing top 5 rows


In [8]:
#Muestra los datos verticalmente, como tarjetas
spark.sql(sql_str).show(5,40,True)

-RECORD 0--------------------------------
 Publisher | Nintendo                    
 TotalNA   | 817.0                       
 TotalGl   | 1787.0                      
-RECORD 1--------------------------------
 Publisher | Electronic Arts             
 TotalNA   | 595.0                       
 TotalGl   | 1110.0                      
-RECORD 2--------------------------------
 Publisher | Activision                  
 TotalNA   | 430.0                       
 TotalGl   | 727.0                       
-RECORD 3--------------------------------
 Publisher | Sony Computer Entertainment 
 TotalNA   | 265.0                       
 TotalGl   | 607.0                       
-RECORD 4--------------------------------
 Publisher | Ubisoft                     
 TotalNA   | 253.0                       
 TotalGl   | 475.0                       
only showing top 5 rows


In [9]:
#Listado de columnas
spark.sql(sql_str).columns

['Publisher', 'TotalNA', 'TotalGl']

In [10]:
#Centa los registros
spark.sql(sql_str).count()

579

-----------------------

### Partición de Datos

In [11]:
#Implementacion de PartitionBy
spark = SparkSession.builder.appName('Partitionby() PySpark').getOrCreate()

#Lee el dataframe elarchivo vgsales nuevamente
dataframe = spark.read.option('header', True).csv('databses/vgsales.csv')

#Imprime el esquema del archivo
dataframe.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



#### PartitionBy()
 - Generando pedazos de informacion


In [12]:
#Usando partitionBy - con la columna platform
#Esto genera un directorio para cada plataforma
dataframe.write.option('header', True).partitionBy('Platform').mode('overwrite').csv('databses/vgsales')

In [13]:
#Usando partitionBy - con la columna year
#Esto genera un directorio para cada plataforma
dataframe.write.option('header', True).partitionBy('Year').mode('overwrite').csv('databses/vgsales/year wtf')

In [14]:
#Usando partitionBy - con la columna year
#Esto genera un directorio para cada plataforma
dataframe.write.option('header', True).partitionBy('Genre','Year').mode('overwrite').csv('databses/vgsales/genre x year')

In [15]:
# Otro ejemplo pero ahora contorlando el numerode particiones (resgistros dentro del archivo)
# 2 lineas -  3 min, 10 lineas - 28 seg , 50 lineas
dataframe.write.option('header',True).option('maxRecordsPerFile',10).partitionBy('Genre').mode('overwrite').csv('databses/vgsales')

### Repartition

In [16]:
#Implemetacion 
spark = SparkSession.builder.appName('coalesce() PySpark').getOrCreate()

#Lee en el dataframe el archivo vgsales
dataco = spark.read.option('header', True).csv('databses/vgsales.csv')

#Imprime el esquema de el archivo
dataco.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



In [20]:
#Repartition
#Generar el directorio el numeo de archivos señalados(20)
dataco.repartition(20).write.mode('overwrite').option('header', True).csv('databses/vgsales/vg')

In [21]:
#Como se explico se puede cambiar el numero de reparticiones y lo hara inmediatamente
dataco.repartition(30).write.mode('overwrite').option('header',True).csv('databses/vgsales/vg')

In [22]:
#nuevo ejemplo con 10 particiones, el directorio se genera automaticamente
dataco.repartition(10).write.mode('overwrite').option('header', True).csv('databses/vgsales/ej10')

In [23]:
dataco.show()

+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|   11.27|    8.89|   10.22|  

In [25]:
#Usando Coalesce
#A diferencia de raepartition, que ejecuta un shuffle para rearmar el # de particiones, coalesce sirve para disminuir el numero de las mismas 
#Coalesce no aplica un full reshuffle a menos que sea necesario
#menos pesado computacionalmente

#Se obtiene el numero de particiones por defecto
dataco_2 = dataco.repartition(20)
dataco_2.rdd.getNumPartitions()

20

In [26]:
dataco_3 = dataco_2.coalesce(10)

In [27]:
dataco_3.rdd.getNumPartitions()

10

In [28]:
dataco_3.write.mode('overwrite').option('header',True).csv('databses/vgsales/coalesce')

----------------------

### PyArrow

In [29]:
from pyarrow import csv 
import pyarrow as pa 

In [32]:
archivo = 'databses/vgsales.csv'
tab_vgsales = csv.read_csv(archivo)

In [33]:
#Estructura de vgsales visto desde PyArrow
tab_vgsales

pyarrow.Table
Rank: int64
Name: string
Platform: string
Year: int64
Genre: string
Publisher: string
NA_Sales: double
EU_Sales: double
JP_Sales: double
Other_Sales: double
Global_Sales: double
----
Rank: [[1,2,3,4,5,...,12661,12662,12663,12664,12665],[12666,12667,12668,12669,12670,...,16596,16597,16598,16599,16600]]
Name: [["Wii Sports","Super Mario Bros.","Mario Kart Wii","Wii Sports Resort","Pokemon Red/Pokemon Blue",...,"Hajime no Ippo Portable: Victorious Spirits","Nanostray","CMT Presents: Karaoke Revolution Country","Katekyoo Hitman Reborn! DS: Flame Rumble Mukuro Kyoushuu","Cid to Chocobo no Fushigi na Dungeon: Toki Wasure no Meikyuu DS+"],["Vitamin X to Z","Fatal Frame","Flip's Twisted World","Tom Clancy's Ghost Recon: Advanced Warfighter","Urban Chaos: Riot Response",...,"Woody Woodpecker in Crazy Castle 5","Men in Black II: Alien Escape","SCORE International Baja 1000: The Official Game","Know How 2","Spirits & Spells"]]
Platform: [["Wii","NES","Wii","Wii","GB",...,"PSP","DS",

In [34]:
#Largo de tab_vgsales
len(tab_vgsales)

16598

In [35]:
#Lista de columnas
tab_vgsales.column_names

['Rank',
 'Name',
 'Platform',
 'Year',
 'Genre',
 'Publisher',
 'NA_Sales',
 'EU_Sales',
 'JP_Sales',
 'Other_Sales',
 'Global_Sales']

In [36]:
#Publica esquema
tab_vgsales.schema

Rank: int64
Name: string
Platform: string
Year: int64
Genre: string
Publisher: string
NA_Sales: double
EU_Sales: double
JP_Sales: double
Other_Sales: double
Global_Sales: double

In [37]:
#mas funciones para descripcion del dataset
tab_vgsales.num_columns

11

In [38]:
#Mas fucniones para descripcion del dataset
tab_vgsales.num_rows

16598

In [39]:
#Para la estructura a Pandas
#Documentacion: Pendiente
df= tab_vgsales.to_pandas()

In [40]:
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [41]:
#Agrupa la tabla por Genero
tab_genre = tab_vgsales.group_by('Genre').aggregate([('NA_Sales','sum')])

In [ ]:
#Pasar resultado a pandas
df1 = tab_genre.to_pandas()

In [43]:
df1.head()

,Genre,NA_Sales_sum
0,Sports,683.35
1,Platform,447.05
2,Racing,359.42
3,Role-Playing,327.28
4,Puzzle,123.78


In [44]:
#Añade una columna con append_column
tab2 = tab_vgsales.append_column('test', pa.array(['0'] * len(tab_vgsales), pa.string()))
df2 = tab2.to_pandas()
df2.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,test
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,0
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,0
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,0
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,0
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,0


--------------

### Formato Parquet
 - Formato de Lineas y columnas

In [45]:
import pyarrow.parquet as pq
import pandas as pd 
import pyarrow as pa 

df = pd.DataFrame({'lin1': [-20,100,200],
                   'lin2': ['este','es un','ejemplo'],
                   'lin3': [False,True,True]},
                   index = list('abc'))

In [46]:
df

,lin1,lin2,lin3
a,-20,este,False
b,100,es un,True
c,200,ejemplo,True


In [53]:
tab_example = pa.Table.from_pandas(df)
pq.write_table(tab_example,'databses/parquet/ejemplo')

In [63]:
#Leyendo el archivo parquet
tab_2 = pq.read_table('databses/parquet/ejemplo')   # pyarrow
tab_2.to_pandas()

,lin1,lin2,lin3
a,-20,este,False
b,100,es un,True
c,200,ejemplo,True
